# Topic 15 - Decorators

Build decorators from first principles: closures, wrapper functions, `functools.wraps`, parameterised decorators, and stacking. No Copilot.

## Learning objectives

- explain how a decorator transforms a function
- write a decorator that takes arguments (decorator factory)
- use `functools.wraps` to preserve metadata
- stack multiple decorators and predict execution order
- connect to FastAPI route decorators

## 1. Functions as first-class objects

A function is a value — can be assigned, passed, returned.

In [ ]:
def greet(name: str) -> str:
    return f"Hello, {name}"

# Assign to variable
g = greet
print(g("Alice"))

# Pass as argument
def call_twice(func, arg):
    return func(arg) + " | " + func(arg)

print(call_twice(greet, "Bob"))

# Return from function
def make_greeter(prefix: str):
    def greeter(name: str) -> str:
        return f"{prefix}, {name}"
    return greeter

hello = make_greeter("Hello")
hi = make_greeter("Hi")
print(hello("Alice"))
print(hi("Bob"))

## 2. Closures — the mechanism behind decorators

In [ ]:
def make_multiplier(factor: int):
    """Returns a function that multiplies by factor."""
    def multiplier(x: int) -> int:
        return x * factor  # 'factor' captured from enclosing scope
    return multiplier

times3 = make_multiplier(3)
times5 = make_multiplier(5)

print(f"times3(4) = {times3(4)}")
print(f"times5(4) = {times5(4)}")

# Each returned function has its own closure
print(f"times3 closure: {times3.__closure__[0].cell_contents}")
print(f"times5 closure: {times5.__closure__[0].cell_contents}")

## 3. Basic decorator: function → function

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print(f"Before {func.__name__}")
        result = func(*args, **kwargs)
        print(f"After {func.__name__}")
        return result
    return wrapper

@my_decorator
def say_hello(name: str) -> str:
    return f"Hello, {name}"

# Equivalent to: say_hello = my_decorator(say_hello)
print(say_hello("Alice"))

## 4. `functools.wraps` — preserving function identity

In [ ]:
from functools import wraps

def decorator_without_wraps(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

def decorator_with_wraps(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@decorator_without_wraps
def func_a():
    """Docstring A"""
    pass

@decorator_with_wraps
def func_b():
    """Docstring B"""
    pass

print(f"Without wraps: name={func_a.__name__}, doc={func_a.__doc__}")
print(f"With wraps: name={func_b.__name__}, doc={func_b.__doc__}")

## 5. Decorators that take arguments (decorator factories)

In [ ]:
def repeat(times: int):
    """Decorator factory: returns a decorator that repeats the call."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            results = []
            for _ in range(times):
                results.append(func(*args, **kwargs))
            return results
        return wrapper
    return decorator

@repeat(3)
def say_hello(name: str) -> str:
    return f"Hello, {name}"

print(say_hello("Alice"))

# Three layers:
# 1. repeat(3) -> returns decorator
# 2. decorator(say_hello) -> returns wrapper
# 3. wrapper replaces say_hello

## 6. Stacking decorators

Bottom decorator applied first (closest to function).

In [ ]:
def decorator_a(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("A: before")
        result = func(*args, **kwargs)
        print("A: after")
        return result
    return wrapper

def decorator_b(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("B: before")
        result = func(*args, **kwargs)
        print("B: after")
        return result
    return wrapper

@decorator_a
@decorator_b
def greet(name: str) -> str:
    print(f"  greet called with {name}")
    return f"Hello, {name}"

print("Calling greet:")
result = greet("Alice")
print(f"Result: {result}")

# Order: decorator_b applied first, then decorator_a
# Execution: A:before -> B:before -> greet -> B:after -> A:after

## 7. Class-based decorators

In [ ]:
from functools import wraps

class CountCalls:
    def __init__(self, func):
        self.func = func
        self.count = 0
        wraps(func)(self)  # Copy metadata to instance
    
    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"Call {self.count} to {self.func.__name__}")
        return self.func(*args, **kwargs)

@CountCalls
def greet(name: str) -> str:
    return f"Hello, {name}"

print(greet("Alice"))
print(greet("Bob"))
print(greet("Carol"))
print(f"Total calls: {greet.count}")

## 8. Practical patterns: retry with exponential backoff

In [ ]:
# Jupyter already runs an event loop, so use top-level await (not asyncio.run()).
import asyncio
import random
from functools import wraps
from typing import Callable, TypeVar

T = TypeVar("T")

def retry(max_attempts: int = 3, base_delay: float = 0.1):
    def decorator(func: Callable[..., T]) -> Callable[..., T]:
        @wraps(func)
        async def wrapper(*args, **kwargs) -> T:
            last_exception = None
            for attempt in range(max_attempts):
                try:
                    return await func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    if attempt < max_attempts - 1:
                        delay = base_delay * (2 ** attempt)
                        print(f"  Attempt {attempt+1} failed: {e}. Retrying in {delay}s...")
                        await asyncio.sleep(delay)
            raise last_exception
        return wrapper
    return decorator

# Simulated flaky function
attempt_count = 0

@retry(max_attempts=3, base_delay=0.05)
async def flaky_call() -> str:
    global attempt_count
    attempt_count += 1
    if attempt_count < 3:
        raise ValueError(f"Simulated failure {attempt_count}")
    return "Success!"

async def main():
    global attempt_count
    attempt_count = 0
    result = await flaky_call()
    print(f"Final result: {result}")

await main()

## 9. Practical patterns: timing decorator

In [ ]:
import time
from functools import wraps

def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timed
def slow_function(n: int) -> int:
    total = 0
    for i in range(n):
        total += i * i
    return total

result = slow_function(1_000_000)
print(f"Result: {result}")

## 10. Practical patterns: Pydantic validation decorator

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from functools import wraps

class InvoiceInput(BaseModel):
    invoice_number: str
    total: float = Field(gt=0)
    vendor: str | None = None

def validate_input(model_class: type[BaseModel]):
    def decorator(func):
        @wraps(func)
        def wrapper(data: dict):
            validated = model_class.model_validate(data)
            return func(validated)
        return wrapper
    return decorator

@validate_input(InvoiceInput)
def process_invoice(invoice: InvoiceInput) -> float:
    return invoice.total * 1.2

# Valid input
print(f"Valid: {process_invoice({'invoice_number': 'INV-001', 'total': 100.0})}")

# Invalid input - caught at boundary
try:
    process_invoice({'invoice_number': 'INV-002', 'total': -50})
except ValidationError as e:
    print(f"Validation error: {e.errors()[0]['msg']}")

## 11. FastAPI connection

Route decorators register the function and return it (or wrapped version).

In [ ]:
# Simplified FastAPI-style router
class SimpleRouter:
    def __init__(self):
        self.routes = []
    
    def get(self, path: str):
        def decorator(func):
            self.routes.append(("GET", path, func))
            return func  # Return original function
        return decorator

app = SimpleRouter()

@app.get("/items/{item_id}")
async def read_item(item_id: int) -> dict:
    return {"item_id": item_id}

@app.get("/users/{user_id}")
async def read_user(user_id: int) -> dict:
    return {"user_id": user_id}

print(f"Registered routes: {app.routes}")

## 12. Completion exercises

Do these by hand. No autocomplete.

In [ ]:
# EXERCISE 1: Write a decorator that logs function entry/exit
# with arguments and return value (truncate long values)

# EXERCISE 2: Write a parameterised decorator `rate_limit(calls_per_second)`
# that enforces a minimum delay between calls

# EXERCISE 3: Create a decorator `cache_result` that memoizes
# function results based on arguments (use functools.lru_cache as reference)

# EXERCISE 4: Predict the output of this stacking:
# @decorator_a
# @decorator_b
# @decorator_c
# def func(): pass
# func()
# Where decorator_x prints "X:before" before call and "X:after" after

print("Exercises ready — implement by hand.")